# Granger-causality analysis negative control
This notebook runs a Granger-causality analysis as a negative control for the main analysis pipeline. The goal is to confirm that any causal relationships detected between time series in the real data (DVG NGS read counts and plaque assay measurements) reflect genuine temporal dependencies rather than artifacts of the test itself. To build the negative control, we destroy the temporal structure of the measured data in two complementary ways: (1) bootstrapped min–max sampling, where each timepoint is replaced by a value drawn uniformly between the observed minimum and maximum across series at that timepoint, and (2) per-series shuffling, where the values within each individual time series are randomly permuted across time. Both procedures preserve the marginal distribution of the original data while eliminating any real temporal ordering, so a well-behaved Granger test should return few or no significant causal links. The plaque assay series is then concatenated back in, the data is log10-transformed and made stationary, and the same Granger pipeline is applied — letting us quantify the false-positive rate and use it as a baseline against the results from the unshuffled data.

**Structure.** The two controls (`shuffled` and `random`) are analysed **separately, end to end**: stationarity, Granger matrices, BH correction, classification, summaries, model fits and swarmplots all run per dataset. Every intermediate result lives in the `neg` registry (`neg[dataset][key][lag]`) instead of in loose globals, so the two branches can never overwrite each other. To add a third control, append it to `DATASETS` and register its dataframe — no other cell needs to change.

**Table of Contents**
- [Granger-causality analysis negative control](#Granger-causality-analysis-negative-control)
- [Data Preparation](#Data-Preparation)
  - [create random ts data](#create-random-ts-data)
  - [Log10 transformation](#Log10-transformation)
  - [Stationarity](#Stationarity)
    - [iterative method](#iterative-method)
- [Granger-causality test](#Granger-causality-test)
  - [Make Granger-causality matrix](#Make-Granger-causality-matrix)
  - [Summarize](#Summarize)
- [Plot fitted models](#Plot-fitted-models)
  - [Functions](#Functions)
    - [Performance metrics](#Performance-metrics)
    - [Plot single fits](#Plot-single-fits)
  - [Fitting and plotting](#Fitting-and-plotting)
    - [all](#all)
    - [corrected](#corrected)
  - [plot single candidates](#plot-single-candidates)
- [Swarmplots](#Swarmplots)
  - [Helper Functions](#Helper-Functions)
  - [Functions](#Functions)
  - [all](#all)
  - [corrected](#corrected)
- [group sizes with increasing lag](#group-sizes-with-increasing-lag)
  - [all](#all)
  - [corrected](#corrected)
  - [all vs corrected](#all-vs-corrected)
- [SSR with increasing lag](#SSR-with-increasing-lag)
  - [corrected](#corrected)


In [1]:
import json, re

with open('./02_granger_causality_analysis_negative_control.ipynb') as f:
    nb = json.load(f)

toc = []
for cell in nb['cells']:
    if cell['cell_type'] != 'markdown':
        continue
    for line in cell['source']:
        m = re.match(r'^(#{1,6})\s+(.*)', line)
        if m:
            level = len(m.group(1))
            title = m.group(2).strip()
            anchor = re.sub(r'[^\w\- ]', '', title).strip().replace(' ', '-')
            toc.append(f"{'  ' * (level-1)}- [{title}](#{anchor})")

# print('\n'.join(toc))

In [2]:
from utils.dip_utils import *
from utils.dip_visuals import *

import matplotlib.pyplot as plt
import numpy as np

from statsmodels.tsa.stattools import adfuller

import warnings
warnings.filterwarnings('ignore')

In [3]:
plt.rcParams['font.size'] = 20
plt.rcParams['font.family'] = 'Arial'

# Data Preparation

In [4]:
# ----------------------------- CONFIG ---------------------------------------
import os

output_prefix = 'data/outputs/plus1_log10_random_shuffled'
os.makedirs(output_prefix, exist_ok=True)
os.makedirs(f'{output_prefix}/plots', exist_ok=True)

# The two negative controls, analysed SEPARATELY end-to-end.
DATASETS = ['shuffled', 'random']

DATASET_TITLES = {
    'shuffled': 'Shuffled',
    'random'  : 'Random (bootstrapped min-max)',
}

# Everything downstream lives in this registry: neg[dataset][key]
# -> no more leaked loop variables, no more accidental cross-talk between
#    the shuffled and the random branch.
neg = {ds: {} for ds in DATASETS}
# ----------------------------------------------------------------------------


In [5]:
output_prefix_actual_data = 'data/outputs/plus1_log10_linear_imputation'
ts_data = pd.read_csv(f'{output_prefix_actual_data}/lin_interpolated_ts_df.csv', index_col=0)
max_fixed_lag = 4

In [6]:
dvg_ts_data = ts_data[ts_data.columns[1:]].copy()
dvg_ts_data

## Create the two negative-control datasets


In [7]:
n_cols =  dvg_ts_data.shape[1]
np.random.seed(42)

fig,axs = plt.subplots(figsize=(12, 5), ncols=2)
ax1,ax2 = axs[0],axs[1]

# get per-time min and max (customize if you already have these)
min_vals = dvg_ts_data.min(axis=1)
max_vals = dvg_ts_data.max(axis=1)

# generate uniformly distributed samples between min and max
ts_data_random = pd.DataFrame(
    np.random.uniform(min_vals.values[:, None], max_vals.values[:, None], size=(len(dvg_ts_data), n_cols)),
    index=dvg_ts_data.index,
    columns=[f'rand_{i+1}' for i in range(n_cols)]
)

ts_data_random.sample(min(250, n_cols), axis=1).plot(ax=ax1)
ax1.set_ylabel('NGS read counts (abs)')
ax1.legend().remove()
ax1.set_title('simulated data (bootstrapped min-max sampling)')

dvg_ts_data.plot(ax=ax2)
ax2.set_ylabel('NGS read counts (abs)')
ax2.legend().remove()
ax2.set_title('Measured data')
plt.tight_layout()
plt.show()


In [8]:
plt.rcParams['font.size'] = 20
np.random.seed(42)
fig,axs = plt.subplots(figsize=(12, 5), ncols=2)
ax1,ax2 = axs[1],axs[0]
ts_data_shuffled = dvg_ts_data.apply(lambda x: np.random.permutation(x.values))
ts_data_shuffled.columns = [f'{col}_shuf' for col in ts_data_shuffled.columns]
dvg_ts_data.plot(ax=ax2)
ax2.set_ylabel('NGS read counts')
ax2.set_xlabel('Time post infection (days)')
ax2.set_title('Raw data\n(measured)')
ax2.legend().remove()
ts_data_shuffled.plot(ax=ax1)
ax1.set_ylabel('NGS read counts')
ax1.set_xlabel('Time post infection (days)')
ax1.legend().remove()
ax1.set_title('Shuffled negative control\n(random permutations of raw data)')
plt.tight_layout()

In [9]:
ts_data_random = pd.concat([ts_data['plaque_assay'], ts_data_random], axis=1)
ts_data_random

In [10]:
ts_data_shuffled = pd.concat([ts_data['plaque_assay'], ts_data_shuffled], axis=1)
ts_data_shuffled

In [11]:
ts_data_arima = pd.concat([ts_data_shuffled,dvg_ts_data], axis=1)
# log-transform
ts_data_arima = np.log10(ts_data_arima + 1)
ts_data_arima.loc[[2.02, 2.49, 2.95, 5.99, 6.49, 6.96, 7.42, 8.42, 9.97, 10.43,
                  10.96, 11.42, 11.97, 13.99, 14.44, 14.97, 15.45, 16.48, 18.45, 18.98]] = np.nan
ts_data_arima.to_csv(f'{output_prefix}/log_transformed_arima_data.csv')
print(f"Saved linear interpolated + log10 transformed data to {output_prefix}/log_transformed_arima_data.csv")

### Register both datasets

`neg` is the single source of truth from here on: `neg[dataset]` holds the time series, the stationarity result, the Granger matrices, the summaries and the model fits for that dataset only.


In [12]:
# Register both control datasets, persist them, and record non-zero counts.
neg['shuffled']['ts'] = ts_data_shuffled
neg['random']['ts']   = ts_data_random

for ds in DATASETS:
    D = neg[ds]
    # DVG columns only (plaque_assay is the cultivation variable, not a DVG)
    D['dvg_cols'] = [c for c in D['ts'].columns if c != 'plaque_assay']
    D['ts'].to_csv(f'{output_prefix}/{ds}_lin_interpolated_ts_data.csv')
    # colour reference used later in the swarmplots
    D['nonzero_counts'] = {k: v for k, v in (D['ts'] > 0).sum().items()}
    print(f"{ds:9s}: {D['ts'].shape[0]} timepoints x {len(D['dvg_cols'])} DVGs")

# kept for backwards-compatibility with any cell that still expects the merged dict
dvg2nonzero_counts = {k: v for ds in DATASETS for k, v in neg[ds]['nonzero_counts'].items()}


## Log10 transformation

In [13]:
for ds in DATASETS:
    # log10(x + 1); avoids the deprecated DataFrame.applymap
    neg[ds]['log_ts'] = np.log10(neg[ds]['ts'] + 1)

neg['shuffled']['log_ts'].head()


## Stationarity

Time-series analysis relies on stationarity so that the models and predictions can be statistically relevant and to avoid the detection of spurious relationships.
Thus we need to pre process the imported raw data. I explored two methods to obtain stationarity:
1. The iterative method
-> repeatedly form the difference of one value at index i with its preceeding value i-1 in the input array

2. The direct method 
-> directly determine which increment is needed to get a stationary time-series (determine x if np.diff(x) gives us a stationary dataframe) (not implemented yet)

### iterative method

In [15]:
import types

def make_stationary(input_df, max_diff=10, autolag=None, max_lag=None, regression="c"):
    """Transform each column (candidate) to a stationary series by iterative differencing.

    Args:
        input_df (pd.DataFrame): rows are timepoints; columns are candidates/variables.
        max_diff (int): maximum differencing level. Defaults to 10.
        autolag (str | None): lag selection for ADF. Defaults to 'AIC'.
        max_lag (int | None): max lags for ADF. Defaults to sqrt(n_timepoints), then clamped.
        regression (str): ADF regression trend ('c', 'ct', 'ctt', 'nc'). Defaults to 'c'.

    Returns:
        df_stationary (pd.DataFrame): final stationary series (may be differenced) for stationary columns.
        df_nonstationary (pd.DataFrame): original non-stationary columns.
        df_diff_nonstationary (pd.DataFrame): last differenced data for non-stationary columns (np.diff, padded).
        diff_levels (pd.Series): differencing level used per column (NaN for non-stationary/error).
        lags (pd.Series): ADF-selected lag per column (NaN for non-stationary/error).
        pvals (pd.Series): final ADF p-value per column (NaN on error).
        status (pd.Series): "stationary", "non-stationary", or "error".
    """
    df = input_df.copy()
    n_timepoints = len(df)

    if autolag is None:
        autolag = "AIC"
    if max_lag is None:
        max_lag = int(np.sqrt(max(n_timepoints, 1)))
    # Clamp to feasible range
    max_lag = max(1, min(max_lag, max(1, n_timepoints // 2 - 1)))

    stationary_data = {}
    nonstationary_cols = []
    diff_nonstationary_dict = {}
    diff_levels = {}
    lags = {}
    pvals = {}
    status = {}

    for col in df.columns:
        series = pd.to_numeric(df[col], errors="coerce")
        # Drop NaNs for testing but keep original index for reconstruction
        ts_data = series.dropna().to_numpy()
        if len(ts_data) < 3 or len(ts_data) <= max_lag:
            # Too short for ADF with this max_lag
            nonstationary_cols.append(col)
            diff_nonstationary_dict[col] = np.diff(series.to_numpy()) if len(series) > 1 else np.array([])
            diff_levels[col] = np.nan
            lags[col] = np.nan
            pvals[col] = np.nan
            status[col] = "error"
            continue

        diff_level = 0
        last_data = ts_data.copy()

        def run_adf(arr):
            with np.errstate(divide='ignore', invalid='ignore'):
                return adfuller(arr, autolag=autolag, regression=regression, maxlag=max_lag)

        # Initial ADF
        try:
            adf_result = run_adf(last_data)
            p_value = adf_result[1]
        except Exception:
            nonstationary_cols.append(col)
            diff_nonstationary_dict[col] = np.diff(series.to_numpy()) if len(series) > 1 else np.array([])
            diff_levels[col] = np.nan
            lags[col] = np.nan
            pvals[col] = np.nan
            status[col] = "error"
            continue

        if p_value <= 0.05:
            # Already stationary; keep original cleaned series, realign to original index
            stationary_series = series
            stationary_data[col] = stationary_series
            diff_levels[col] = diff_level
            lags[col] = adf_result[2]
            pvals[col] = p_value
            status[col] = "stationary"
            continue

        # Iterative differencing
        became_stationary = False
        while diff_level < max_diff:
            diff_data = np.diff(last_data)
            diff_level += 1
            # Need at least 3 points to run ADF reasonably
            if len(diff_data) < 3 or len(diff_data) <= max_lag:
                # Can't test further; mark non-stationary
                break
            try:
                adf_result = run_adf(diff_data)
                p_value = adf_result[1]
            except Exception:
                break

            if p_value <= 0.05:
                # Store differenced series aligned to original index (pad front with NaN)
                padded = np.concatenate(([np.nan] * diff_level, diff_data))
                # Trim/pad to original length
                padded = padded[:n_timepoints] if len(padded) >= n_timepoints else np.pad(
                    padded, (0, n_timepoints - len(padded)), constant_values=np.nan
                )
                stationary_series = pd.Series(padded, index=df.index)
                stationary_data[col] = stationary_series
                diff_levels[col] = diff_level
                lags[col] = adf_result[2]
                pvals[col] = p_value
                status[col] = "stationary"
                became_stationary = True
                break

            last_data = diff_data

        if not became_stationary:
            nonstationary_cols.append(col)
            # Keep the last differenced data for inspection
            diff_nonstationary_dict[col] = np.diff(series.to_numpy()) if len(series) > 1 else np.array([])
            diff_levels[col] = np.nan
            lags[col] = np.nan
            pvals[col] = np.nan
            status[col] = "non-stationary"

    # Assemble outputs
    df_stationary = pd.DataFrame(stationary_data, index=df.index) if stationary_data else pd.DataFrame(index=df.index)
    df_nonstationary = df[nonstationary_cols].copy() if nonstationary_cols else pd.DataFrame(index=df.index)

    if diff_nonstationary_dict:
        padded = {}
        for col, arr in diff_nonstationary_dict.items():
            # np.diff gives length n-1; pad with leading NaN to align to n
            padded_arr = np.concatenate(([np.nan], arr))
            if len(padded_arr) < n_timepoints:
                padded_arr = np.pad(padded_arr, (0, n_timepoints - len(padded_arr)), constant_values=np.nan)
            else:
                padded_arr = padded_arr[:n_timepoints]
            padded[col] = padded_arr
        df_diff_nonstationary = pd.DataFrame(padded, index=df.index)
    else:
        df_diff_nonstationary = pd.DataFrame(index=df.index)

    return types.SimpleNamespace(
        df_stationary=df_stationary,
        df_nonstationary=df_nonstationary,
        df_diff_nonstationary=df_diff_nonstationary,
        diff_levels=pd.Series(diff_levels),
        lags=pd.Series(lags),
        pvals=pd.Series(pvals),
        status=pd.Series(status),
    )


In [16]:
for ds in DATASETS:
    D = neg[ds]
    print(f'Making {ds} data stationary...')
    D['stationarity'] = make_stationary(D['log_ts'], max_diff=10, autolag='AIC', max_lag=None)
    D['stationary_ts_df'] = D['stationarity'].df_stationary.fillna(0)
    D['diff_levels'] = D['stationarity'].diff_levels.to_dict()

    n_stat = (D['stationarity'].status == 'stationary').sum()
    n_tot  = len(D['stationarity'].status)
    print(f'  {ds}: {n_stat}/{n_tot} series stationary '
          f"({(D['stationarity'].status == 'non-stationary').sum()} non-stationary, "
          f"{(D['stationarity'].status == 'error').sum()} errors)")

    assert 'plaque_assay' in D['stationary_ts_df'].columns, \
        f'plaque_assay was dropped as non-stationary in the {ds} dataset'


In [17]:
neg['shuffled']['stationary_ts_df'].to_csv(f'{output_prefix}/shuffled_stationary_ts_df.csv')

In [18]:
neg['shuffled']['stationary_ts_df']

# Granger-causality test

In [19]:
# importing the Granger-causality test from statsmodels
from statsmodels.tsa.stattools import grangercausalitytests

# assigning the string 'ssr_chi2test' to the variable 'test'
test = 'ssr_chi2test'
err_dips = {}

In [20]:
def granger_causation_matrix_fixed_lag(data, variables, cultivation_values=[],test='ssr_chi2test', plot=False,fixlag=3, maxlag=3, debug=False):
    test_results = {}
    err_dips = {}
    max_lags = {}
    # creating a dataframe with the same dimensions as number of variables entered, assigned to the variable 'X_train'
    X_train = pd.DataFrame(np.nan, index=variables, columns=variables)
    
    # loops through the columns and the indexes
    for c in X_train.columns:
        for r in X_train.index:
            #skip if we're not looking at any cultivation variable
            if r == c:
                continue
            go_on = False
            if c in cultivation_values:
                go_on = True
            if r in cultivation_values:
                go_on = True
            
            if go_on == False:
                continue
            # conducts a Granger-causality test on a variable row and column using the 'maxlag' variable; assigns to 
            # variable test_result
            try:
                test_result = grangercausalitytests(data[[r, c]], maxlag=[maxlag], verbose=False)
            except Exception as e:
                if debug:
                  print(f'Error on {r} and {c}, {data[[r, c]]}: {e}')
                err_dips[(r,c)] = e
                X_train.loc[r, c] = None
                max_lags[(r,c)] = None
                continue
            test_results[(r,c)] = test_result    
            # locates the test result in the tuple 'test_result' and rounds the number by 4 digits; assigns to 'p_values'
            p_value = round(test_result[fixlag][0][test][1], 4)
            max_lags[(r,c)] = fixlag
            X_train.loc[r, c] = p_value
            
    # rename the row and column names based on the relationship
    X_train.columns = [var + '_x' for var in variables]
    X_train.index = [var + '_y' for var in variables]
    return X_train, test_results, max_lags, err_dips

In [21]:
from statsmodels.stats.multitest import multipletests

import numpy as np
from statsmodels.stats.multitest import multipletests

import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

def correct_p_values_bh_separately(matrix, cult_columns=5, alpha=0.05, debug=False):
    """
    Applies Benjamini-Hochberg correction cultivation-column-wise:
    1. DVGs -> each Virus Concentration column (row i, DVG columns to the right)
    2. Each Virus Concentration column -> DVGs (DVG rows, column i)

    Parameters:
        matrix (pd.DataFrame): A pandas DataFrame where each cell contains a p-value.
        cult_columns (int): Number of columns corresponding to virus concentration variables.
        alpha (float): significance level for BH.
        debug (bool): if True, print and display slices being corrected.

    Returns:
        pd.DataFrame: Same shape with BH-adjusted p-values.
    """
    corrected_matrix = matrix.copy()

    for col_i in range(cult_columns):
        col_name = matrix.columns[col_i].replace("_x", "")

        # DVG -> col_i (row col_i, DVG columns to the right)
        dvg_to_col = matrix.iloc[col_i, cult_columns:].values
        if debug:
            print(f"\n[DVG -> {col_name}] slice BEFORE correction:")
            display(matrix.iloc[col_i:col_i+1, cult_columns:])
        mask = ~np.isnan(dvg_to_col)
        if mask.any():
            _, corrected_vals, _, _ = multipletests(dvg_to_col[mask], alpha=alpha, method="fdr_bh")
            tmp = dvg_to_col.copy()
            tmp[mask] = corrected_vals
            corrected_matrix.iloc[col_i, cult_columns:] = tmp
            if debug:
                print(f"[DVG -> {col_name}] slice AFTER correction:")
                display(corrected_matrix.iloc[col_i:col_i+1, cult_columns:])

        # col_i -> DVGs (all DVG rows below, column col_i)
        col_to_dvg = matrix.iloc[cult_columns:, col_i].values
        if debug:
            print(f"\n[{col_name} -> DVGs] slice BEFORE correction:")
            display(matrix.iloc[cult_columns:, col_i:col_i+1])
        mask = ~np.isnan(col_to_dvg)
        if mask.any():
            _, corrected_vals, _, _ = multipletests(col_to_dvg[mask], alpha=alpha, method="fdr_bh")
            tmp = col_to_dvg.copy()
            tmp[mask] = corrected_vals
            corrected_matrix.iloc[cult_columns:, col_i] = tmp
            if debug:
                print(f"[{col_name} -> DVGs] slice AFTER correction:")
                display(corrected_matrix.iloc[cult_columns:, col_i:col_i+1])
    return corrected_matrix

def _bh_critical_pval(pvals, alpha=0.05, debug=False):
    """Return the BH critical p-value for a 1D array (ignoring NaNs)."""
    p = np.sort(np.asarray(pvals)[~np.isnan(pvals)])
    m = p.size
    if m == 0:
        if debug:
            print("No valid p-values found.")
        return None
    thresholds = alpha * (np.arange(1, m+1) / m)
    hits = np.where(p <= thresholds)[0]
    if debug:
      # create df
      tmp_df = pd.DataFrame({'p-values': p, 'thresholds': thresholds})
      tmp_df['hits'] = np.where(p <= thresholds, 'hit', 'miss')
      display(tmp_df)
    if hits.size == 0:
      if debug:
          print("No hits found.")
      return None
    kmax = hits.max()
    if debug:
      print(f"Critical p-value for BH correction at rank {kmax}: {p[kmax]}")
    return p[kmax]   # the largest p-value that still passes

def calculate_critical_pval_bh(matrix, cult_columns=5, alpha=0.05, debug=False):
    """
    For each cultivation column i (0..cult_columns-1), return:
      - 'dvg_to_<colname>': BH critical p for DVGs -> <colname>
        (row i, DVG columns to the right)
      - '<colname>_to_dvg': BH critical p for <colname> -> DVGs
        (DVG rows below, column i)
    """
    crit_dct = {}
    for i, col_name in enumerate(matrix.columns[:cult_columns]):
        clean = col_name.replace('_x', '')

        # DVG -> col_i  (row i, DVG columns)
        dvg_to_col_slice = matrix.iloc[i, cult_columns:]
        if debug:
            print(f"\n[DVG -> {clean}] slice:")
            display(dvg_to_col_slice.to_frame().T)
        dvg_to_col = dvg_to_col_slice.values
        crit_dct[f"dvg_to_{clean}"] = _bh_critical_pval(dvg_to_col, alpha, debug)

        # col_i -> DVGs  (DVG rows, column i)
        col_to_dvg_slice = matrix.iloc[cult_columns:, i]
        if debug:
            print(f"\n[{clean} -> DVGs] slice:")
            display(col_to_dvg_slice.to_frame())
        col_to_dvg = col_to_dvg_slice.values
        crit_dct[f"{clean}_to_dvg"] = _bh_critical_pval(col_to_dvg, alpha, debug)

    return crit_dct

## Make Granger-causality matrix

How to read the matrix: column X Granger-causes row Y
i.e. X improves the forecasting performance of Y if included in an OLS model

In [22]:
import pickle

# per-dataset result containers -- all keyed by fixed lag
for ds in DATASETS:
    neg[ds].update({
        'gc_matrix'          : {},
        'test_results'       : {},
        'gc_max_lag'         : {},
        'errors'             : {},   # was shared between both branches before
        'corrected_gc_matrix': {},
        'bh_critical_p_vals' : {},
    })


In [23]:
for ds in DATASETS:
    D = neg[ds]
    stationary_ts_df = D['stationary_ts_df']

    for fix_lag in range(1, max_fixed_lag):
        print(f'[{ds}] Calculating Granger-causality matrix for fixed lag {fix_lag}...')

        (D['gc_matrix'][fix_lag],
         D['test_results'][fix_lag],
         D['gc_max_lag'][fix_lag],
         D['errors'][fix_lag]) = granger_causation_matrix_fixed_lag(
            stationary_ts_df,
            variables=stationary_ts_df.columns.tolist(),
            cultivation_values=['plaque_assay'],
            fixlag=fix_lag,
            maxlag=fix_lag,
        )

        D['corrected_gc_matrix'][fix_lag] = correct_p_values_bh_separately(
            D['gc_matrix'][fix_lag], cult_columns=1, alpha=0.05, debug=False,
        )

        D['bh_critical_p_vals'][fix_lag] = calculate_critical_pval_bh(
            D['gc_matrix'][fix_lag], cult_columns=1, alpha=0.05, debug=False,
        )

        n_err = len(D['errors'][fix_lag])
        if n_err:
            print(f'  [{ds}] lag {fix_lag}: {n_err} Granger test(s) failed')


In [24]:
for ds in DATASETS:
    print(f'--- {ds}: critical p-values with BH correction ---')
    for fix_lag, crit in neg[ds]['bh_critical_p_vals'].items():
        print(f'  lag {fix_lag}: {crit}')


### p-value distributions


In [25]:
import math

plt.rcParams.update({'font.size': 8})
n_lags = max_fixed_lag - 1

for ds in DATASETS:
    D = neg[ds]
    for direction, title in [('dvg_to_pfu', 'DVG -> PFU'), ('pfu_to_dvg', 'PFU -> DVG')]:
        cols = min(n_lags, 6)
        rows = math.ceil(n_lags / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(15, 2 * rows),
                                 constrained_layout=True, squeeze=False)
        axes = axes.flatten()

        for i, fixlag in enumerate(range(1, max_fixed_lag)):
            ax = axes[i]
            if direction == 'dvg_to_pfu':
                pvals = D['gc_matrix'][fixlag].iloc[0]        # plaque_assay row
            else:
                pvals = D['gc_matrix'][fixlag].iloc[1:, 0]    # plaque_assay column
            pvals.hist(bins=50, ax=ax)
            ax.set_title(f'lag={fixlag}')
            ax.set_xlabel('p-value')
            ax.set_ylabel('Frequency')

        for j in range(n_lags, len(axes)):
            axes[j].axis('off')

        plt.suptitle(f'{DATASET_TITLES[ds]} - p-values for Granger-causality {title}', fontsize=10)
        plt.show()


## Summarize

In [26]:
def classify_granger_dips(gc_matrix, cultivation_vals):
    granger_caused_dips = {}
    granger_not_caused_dips = {}
    granger_causing_dips = {}
    granger_not_causing_dips = {}
    granger_bi_dips = {}
    non_related_dips = {}
    for cultivation_val in cultivation_vals:
        granger_caused_dips[cultivation_val] = [dip.replace('_y', '') for dip in gc_matrix[gc_matrix[cultivation_val + '_x'] <= 0.05].index]
        granger_not_caused_dips[cultivation_val] = [dip.replace('_y', '') for dip in gc_matrix[gc_matrix[cultivation_val + '_x'] > 0.05].index]
        tmp_y = gc_matrix.loc[cultivation_val + '_y']
        granger_causing_dips[cultivation_val] = [dip.replace('_x', '') for dip in tmp_y[tmp_y <= 0.05].index]
        granger_not_causing_dips[cultivation_val] = [dip.replace('_x', '') for dip in tmp_y[tmp_y > 0.05].index]
        
        granger_bi_dips[cultivation_val] = list(set(granger_causing_dips[cultivation_val]).intersection(set(granger_caused_dips[cultivation_val])))
        non_related_dips[cultivation_val] = list(set(granger_not_caused_dips[cultivation_val]).intersection(set(granger_not_causing_dips[cultivation_val])))
    return granger_caused_dips, granger_causing_dips, granger_bi_dips, non_related_dips
  
def classify_granger_dips_bh(gc_matrix, cultivation_vals, bh_critical_pvals):    
    granger_caused_dips = {}
    granger_not_caused_dips = {}
    granger_causing_dips = {}
    granger_not_causing_dips = {}
    granger_bi_dips = {}
    non_related_dips = {}
    for cultivation_val in cultivation_vals:
        critical_causing_pval = bh_critical_pvals[f"dvg_to_{cultivation_val}"]
        critical_caused_pval = bh_critical_pvals[f"{cultivation_val}_to_dvg"]
        granger_caused_dips[cultivation_val] = [dip.replace('_y', '') for dip in gc_matrix[gc_matrix[cultivation_val + '_x'] <= critical_caused_pval].index]
        granger_not_caused_dips[cultivation_val] = [dip.replace('_y', '') for dip in gc_matrix[gc_matrix[cultivation_val + '_x'] > critical_caused_pval].index]
        tmp_y = gc_matrix.loc[cultivation_val + '_y']
        granger_causing_dips[cultivation_val] = [dip.replace('_x', '') for dip in tmp_y[tmp_y <= critical_causing_pval].index]
        granger_not_causing_dips[cultivation_val] = [dip.replace('_x', '') for dip in tmp_y[tmp_y > critical_causing_pval].index]
        
        granger_bi_dips[cultivation_val] = list(set(granger_causing_dips[cultivation_val]).intersection(set(granger_caused_dips[cultivation_val])))
        non_related_dips[cultivation_val] = list(set(granger_not_caused_dips[cultivation_val]).intersection(set(granger_not_causing_dips[cultivation_val])))
    return granger_caused_dips, granger_causing_dips, granger_bi_dips, non_related_dips
        

In [27]:
for ds in DATASETS:
    neg[ds].update({
        'caused': {}, 'causing': {}, 'bi': {}, 'nonrel': {},
        'caused_bh': {}, 'causing_bh': {}, 'bi_bh': {}, 'nonrel_bh': {},
        'caused_corr': {}, 'causing_corr': {}, 'bi_corr': {}, 'nonrel_corr': {},
    })

for ds in DATASETS:
    D = neg[ds]
    for fix_lag in range(1, max_fixed_lag):
        # 1) raw p-values, hard 0.05 cut-off
        (D['caused'][fix_lag], D['causing'][fix_lag],
         D['bi'][fix_lag], D['nonrel'][fix_lag]) = classify_granger_dips(
            D['gc_matrix'][fix_lag], cultivation_vals=['plaque_assay'],
        )

        # 2) raw p-values, BH critical value as cut-off
        (D['caused_bh'][fix_lag], D['causing_bh'][fix_lag],
         D['bi_bh'][fix_lag], D['nonrel_bh'][fix_lag]) = classify_granger_dips_bh(
            D['gc_matrix'][fix_lag], cultivation_vals=['plaque_assay'],
            bh_critical_pvals=D['bh_critical_p_vals'][fix_lag],
        )

        # 3) BH-adjusted p-values, hard 0.05 cut-off
        (D['caused_corr'][fix_lag], D['causing_corr'][fix_lag],
         D['bi_corr'][fix_lag], D['nonrel_corr'][fix_lag]) = classify_granger_dips(
            D['corrected_gc_matrix'][fix_lag], cultivation_vals=['plaque_assay'],
        )


In [28]:
def create_summary_df(gc_test_results, 
                      gc_max_lag,
                      granger_caused_dips, granger_causing_dips, granger_bi_dips, non_related_dips,
                      diff_level_dct,
                      readcounts_data,
                      time_series_data,
                      output_file, 
                      stat_tests=['ssr_ftest', 'ssr_chi2test', 'lrtest'],
                      cultivation_values=['vrna', 'tcid50', 'plaque_assay', 'log_ha', 'ha_titer']):
    
    # Collect all unique DIPs that were tested (including non-related)
    all_tested_dips = set(
        dip for dips in granger_caused_dips.values() for dip in dips
    ).union(
        dip for dips in granger_causing_dips.values() for dip in dips
    ).union(
        dip for dips in non_related_dips.values() for dip in dips
    )

    # Initialize summary dataframe with filtered keys
    summary_df = readcounts_data[readcounts_data['key'].isin(all_tested_dips)][['key']].copy()
    
    # Assign Granger-causality labels
    for cultivation_value in cultivation_values:
        summary_df[cultivation_value + '_granger_label'] = summary_df['key'].apply(
            lambda dip: (
                'bi-directional' if dip in granger_bi_dips.get(cultivation_value, [])
                else 'causing' if dip in granger_causing_dips.get(cultivation_value, [])
                else 'caused' if dip in granger_caused_dips.get(cultivation_value, [])
                else 'non-related' if dip in non_related_dips.get(cultivation_value, []) 
                else None  # DIPs that were never processed
            )
        )
    
    # Store max_diff level
    summary_df['max_diff'] = summary_df['key'].map(diff_level_dct).fillna(0)
    
    # Function to extract Granger test statistics (Unchanged)
    def extract_granger_stats(key, cultivation_value, direction):
        """Extracts Granger test statistics, lags, and correlation values."""
        tr_key = (cultivation_value, key) if direction == 'causing' else (key, cultivation_value)
        
        if tr_key not in gc_test_results or tr_key not in gc_max_lag:
            return None, None, None, [None] * len(stat_tests)
        
        tr = gc_test_results[tr_key]
        max_lag = gc_max_lag[tr_key]
        
        # Extract p-values for each statistical test
        
        stat_pvals = [tr[max_lag][0][test][1] for test in stat_tests]
        
        # Compute cross-correlation
        cross_corr_value = time_series_data[key].corr(
            time_series_data[cultivation_value].shift(-max_lag)  # Always shift as "causing"
        )
        
        return max_lag, cross_corr_value, stat_pvals
    
    # Process each cultivation variable
    for cultivation_value in cultivation_values:
        max_lags, cross_corr_values, significant_pvals_lags = [], [], []
        stat_tests_dct = {test: [] for test in stat_tests}
        
        for key in summary_df['key']:
            granger_label = summary_df.loc[summary_df['key'] == key, cultivation_value + '_granger_label'].values[0]
            
            # Extract Granger stats for all tested DIPs, including "non-related"
            if granger_label is not None:
                direction = 'causing'  # Always using "causing" for now
                
                max_lag, cross_corr, stat_pvals = extract_granger_stats(key, cultivation_value, direction)
                
                max_lags.append(max_lag)
                cross_corr_values.append(cross_corr)
                
                for i, test in enumerate(stat_tests):
                    stat_tests_dct[test].append(stat_pvals[i])
            else:
                # Skip unprocessed dips
                max_lags.append(None)
                cross_corr_values.append(None)
                for test in stat_tests:
                    stat_tests_dct[test].append(None)
        
        # Assign extracted values to dataframe
        summary_df[cultivation_value + '_max_lag'] = max_lags
        summary_df[cultivation_value + '_cross_corr'] = cross_corr_values
        
        for test in stat_tests:
            summary_df[cultivation_value + '_' + test] = stat_tests_dct[test]
        
        summary_df[cultivation_value + '_significant_pvals'] = summary_df[
            [cultivation_value + '_' + test for test in stat_tests]
        ].lt(0.05).sum(axis=1)
    
    # Compute Granger score
    granger_score_columns = [
        col for cultivation_value in cultivation_values 
        for col in [cultivation_value + '_significant_pvals']
    ]
    summary_df['granger_score'] = summary_df[granger_score_columns].sum(axis=1) - summary_df['max_diff']
    
    # Sort by Granger score and max_diff
    summary_df = summary_df.sort_values(by=['granger_score', 'max_diff'], ascending=[False, True])
    
    # Save to CSV
    summary_df.to_csv(output_file, index=False)

    return summary_df

In [29]:
# one read-count table per dataset (keys = that dataset's DVGs only)
for ds in DATASETS:
    D = neg[ds]
    D['read_counts_df'] = (
        D['stationary_ts_df'].T
        .loc[[c for c in D['stationary_ts_df'].columns if c != 'plaque_assay']]
        .reset_index()
        .rename(columns={'index': 'key'})
    )

neg['shuffled']['read_counts_df'].head()


In [30]:
for ds in DATASETS:
    neg[ds].update({'summary': {}, 'summary_corrected': {}, 'summary_bh': {}})

for ds in DATASETS:
    D = neg[ds]
    for fixlag in range(1, max_fixed_lag):
        print(f'[{ds}] Building summary dataframes for lag {fixlag}...')

        common = dict(
            gc_test_results=D['test_results'][fixlag],
            gc_max_lag=D['gc_max_lag'][fixlag],
            diff_level_dct=D['diff_levels'],          # per-dataset, no longer merged
            readcounts_data=D['read_counts_df'],
            time_series_data=D['stationary_ts_df'],
            stat_tests=['ssr_ftest', 'ssr_chi2test', 'lrtest'],
            cultivation_values=['plaque_assay'],
        )

        D['summary'][fixlag] = create_summary_df(
            granger_causing_dips=D['causing'][fixlag],
            granger_caused_dips=D['caused'][fixlag],
            granger_bi_dips=D['bi'][fixlag],
            non_related_dips=D['nonrel'][fixlag],
            output_file=f'{output_prefix}/{ds}_gc_summary_df_lag{fixlag}.csv',
            **common,
        )

        D['summary_corrected'][fixlag] = create_summary_df(
            granger_causing_dips=D['causing_corr'][fixlag],
            granger_caused_dips=D['caused_corr'][fixlag],
            granger_bi_dips=D['bi_corr'][fixlag],
            non_related_dips=D['nonrel_corr'][fixlag],
            output_file=f'{output_prefix}/{ds}_gc_summary_df_corrected_lag{fixlag}.csv',
            **common,
        )

        D['summary_bh'][fixlag] = create_summary_df(
            granger_causing_dips=D['causing_bh'][fixlag],
            granger_caused_dips=D['caused_bh'][fixlag],
            granger_bi_dips=D['bi_bh'][fixlag],
            non_related_dips=D['nonrel_bh'][fixlag],
            output_file=f'{output_prefix}/{ds}_gc_summary_df_bh_lag{fixlag}.csv',
            **common,
        )


### Time series with Granger labels


In [31]:
# time series + granger label per lag, one table per dataset
for ds in DATASETS:
    D = neg[ds]
    stationary_df = D['stationary_ts_df'].T.copy()

    with_gc_labels = pd.DataFrame(index=stationary_df.index)
    for fixlag in range(1, max_fixed_lag):
        for cultivation_value in ['plaque_assay']:
            for _, row in D['summary'][fixlag].iterrows():
                with_gc_labels.loc[row['key'], f'lag{fixlag}_{cultivation_value}_granger_label'] = \
                    row[cultivation_value + '_granger_label']

    label_cols = [c for c in with_gc_labels.columns if c.endswith('_granger_label')]
    with_gc_labels = pd.concat([with_gc_labels, stationary_df], axis=1)
    with_gc_labels = with_gc_labels.dropna(subset=label_cols, how='all')

    D['with_gc_labels'] = with_gc_labels
    with_gc_labels.to_csv(f'{output_prefix}/{ds}_dvg_time_series_with_gc_labels.csv')
    print(f'[{ds}] labelled time series: {with_gc_labels.shape}')

neg['shuffled']['with_gc_labels'].head()


# Plot fitted models

## Functions

### Performance metrics

In [32]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from tslearn.metrics import dtw, dtw_path

from scipy.signal import argrelextrema
from scipy.stats import ttest_ind, mannwhitneyu, shapiro, pearsonr


In [33]:
def nrsme(y_true, y_pred):
    return root_mean_squared_error(y_true, y_pred, squared=False) / (np.max(y_true) - np.min(y_true))

def mase(y_true, y_pred):
    # Calculate naive forecast (shifted by one time step)
    naive_forecast = np.roll(y_true, shift=1)
    # Remove the first element to align with y_true[1:]
    naive_forecast = naive_forecast[1:]
    
    # Calculate MAE of the model
    mae_model = mean_absolute_error(y_true, y_pred)
    # Calculate MAE of the naive forecast
    mae_naive = mean_absolute_error(y_true[1:], naive_forecast)
    
    return mae_model / mae_naive

def find_local_extrema(y, order=1):
    """
    Find indices of local minima and maxima in the array.
    
    Parameters:
    - y: Array of values.
    - order: How many points on each side to use for the comparison to consider a point a local minimum or maximum.
    
    Returns:
    - minima_indices: Indices of local minima.
    - maxima_indices: Indices of local maxima.
    """
    minima_indices = argrelextrema(y, np.less, order=order)[0]
    maxima_indices = argrelextrema(y, np.greater, order=order)[0]
    
    # Check the first and last points manually
    if len(y) > 1:
        if y[0] < y[1]:
            minima_indices = np.insert(minima_indices, 0, 0)
        if y[0] > y[1]:
            maxima_indices = np.insert(maxima_indices, 0, 0)
        if y[-1] < y[-2]:
            minima_indices = np.append(minima_indices, len(y) - 1)
        if y[-1] > y[-2]:
            maxima_indices = np.append(maxima_indices, len(y) - 1)
    
    return minima_indices, maxima_indices

def calculate_weighted_mae(y_true, y_pred, order=1, weight_factor=3):
    """
    Calculate the Weighted Mean Absolute Error (WMAE) with higher weights on local minima and maxima.
    
    Parameters:
    - y_true: Array of true values.
    - y_pred: Array of predicted values.
    - order: Order for finding local minima and maxima.
    - weight_factor: Weight assigned to local minima and maxima.
    
    Returns:
    - wmae: Weighted Mean Absolute Error.
    """
    y_true_arr = np.array(y_true)
    y_pred_arr = np.array(y_pred)
    # Find local minima and maxima
    minima_indices, maxima_indices = find_local_extrema(y_true_arr, order)
    
    # Initialize weights to 1
    weights = np.ones_like(y_true_arr)
    
    # Assign higher weights to local minima and maxima
    weights[minima_indices] = weight_factor
    weights[maxima_indices] = weight_factor
    
    # Calculate weighted MAE
    wmae = np.sum(weights * np.abs(y_true_arr - y_pred_arr) / np.sum(weights))
    
    return wmae

def calculate_weighted_mape(y_true, y_pred, order=1, weight_factor=3):
    """
    Calculate the Weighted Mean Absolute Percentage Error (WMAPE) with higher weights on local minima and maxima.
    
    Parameters:
    - y_true: Array of true values.
    - y_pred: Array of predicted values.
    - order: Order for finding local minima and maxima.
    - weight_factor: Weight assigned to local minima and maxima.
    
    Returns:
    - wmape: Weighted Mean Absolute Percentage Error.
    """
    y_true_arr = np.array(y_true)
    y_pred_arr = np.array(y_pred)
    # Find local minima and maxima
    minima_indices, maxima_indices = find_local_extrema(y_true_arr, order)
    
    # Initialize weights to 1
    weights = np.ones_like(y_true_arr)
    
    # Assign higher weights to local minima and maxima
    weights[minima_indices] = weight_factor
    weights[maxima_indices] = weight_factor
    
    # Calculate WMAPE
    # To avoid division by zero, we add a small epsilon value to y_true
    epsilon = 1e-10
    wmape = np.sum(weights * np.abs((y_true_arr - y_pred_arr) / (y_true_arr + epsilon))) / np.sum(weights)
    
    return wmape

def dtw_pearson(y_true, y_pred):
    """
    Calculate the Dynamic Time Warping (DTW) aligned Pearson correlation coefficient between two time series.
    
    Parameters:
    - y_true: Array of true values.
    - y_pred: Array of predicted values.
    
    Returns:
    - pearson_corr: The Pearson correlation coefficient between the aligned (warped) y_true and y_pred.
    """
    y_true = np.array(y_true).reshape(-1, 1)
    y_pred = np.array(y_pred).reshape(-1, 1)
    path, dtw_distance = dtw_path(y_true, y_pred)
    aligned_true = np.array([y_true[i] for i, j in path]).flatten()
    aligned_pred = np.array([y_pred[j] for i, j in path]).flatten()
    pearson_corr = pearsonr(aligned_true, aligned_pred)[0]
    return pearson_corr

def dtw_mape(y_true, y_pred, epsilon=1e-6):
    """
    Calculate the DTW-aligned Mean Absolute Percentage Error (MAPE) between two time series.
    
    Parameters:
    - y_true: Array of true values.
    - y_pred: Array of predicted values.
    - epsilon: Small constant to avoid division by zero in MAPE calculation.
    
    Returns:
    - dtw_distance: The DTW distance between y_true and y_pred.
    - dtw_aligned_mape: The MAPE calculated on the DTW-aligned series.
    """
    y_true = np.array(y_true).reshape(-1, 1)
    y_pred = np.array(y_pred).reshape(-1, 1)
    
    # Calculate DTW path and distance
    path, dtw_distance = dtw_path(y_true, y_pred)
    
    # Extract aligned series
    aligned_true = np.array([y_true[i] for i, j in path]).flatten()
    aligned_pred = np.array([y_pred[j] for i, j in path]).flatten()
    
    # Calculate MAPE on aligned series
    dtw_aligned_mape = np.mean(np.abs((aligned_true - aligned_pred) / (aligned_true + epsilon))) * 100
    
    return dtw_aligned_mape



### Plot single fits

In [34]:
granger_label_color_map = {
  'non-related': 'gray',
  'causing': 'tab:blue',
  'caused': 'tab:red',
  'bi-directional': 'tab:purple'
}

def run_granger_prediction(
    summary_df,
    gc_max_lag,
    gc_test_results,
    plot_dips,
    log_long_data,
    restricted_pred_index=None,
    include_ssr=False,
    figsize=(8,6),
    ylabel='log10(PFU)',
    xlabel='Time post infection (days)',
    cultivation_value_label='plaque_assay',
    retransform=False,
    yscale='linear'
):
    # Initialize data structures
    columns = ['key', 'granger_label', 'optimal_lag', 'diff_level', 'ssr_chi2test_pval', 'significant_pvals',
                'mape', 'mae', 'rmse', 'nrsme', 'mse', 'dtw', 'wmae', 'wmape', 'relative_mape',
                'relative_rmse', 'relative_mae', 'relative_wmae', 'relative_wmape', 'relative_dtw',
                'relative_mse', 'dtw_aligned_mape', 'ssr', 'weighted_ssr', 'pred_data']

    gc_prediction_summary_tmp = pd.DataFrame(columns=columns)
    ref_ols_performance_dct_tmp = {}
    full_preds = {}

    # Prepare dataframe
    summary_df_cp = summary_df.copy()
    summary_df_cp[f'{cultivation_value_label}_granger_label'] = summary_df_cp[f'{cultivation_value_label}_granger_label'].fillna('non-related')

    if retransform:
      # retransform all time series data
      log_long_data = np.power(10, log_long_data)
    restricted_pred_done = False
    for idx, row in summary_df_cp.iterrows():
        dip = row.key
        opt_lag = gc_max_lag[(cultivation_value_label, dip)]
        full_pred = gc_test_results[cultivation_value_label, dip][opt_lag][1][1].predict()
        if retransform:
          full_pred = np.power(10, full_pred)
          
        if not restricted_pred_done:
          restricted_pred_index = idx if restricted_pred_index is None else restricted_pred_index
          restricted_pred = gc_test_results[cultivation_value_label, dip][opt_lag][1][0].predict()
          if retransform:
            # Retransform predictions if necessary
            restricted_pred = np.power(10, restricted_pred)
          restricted_pred_done = True
                
        full_preds[dip] = full_pred
        actual_value = log_long_data[cultivation_value_label]
        actual_value_comp = actual_value.iloc[opt_lag:]

        dpis = log_long_data.index.values

        metrics = {
              'mape': mean_absolute_percentage_error(actual_value_comp, full_pred),
              'mae': mean_absolute_error(actual_value_comp, full_pred),
              'mse': root_mean_squared_error(actual_value_comp, full_pred)**2
          }

        metrics['rmse'] = np.sqrt(metrics['mse'])
        metrics['nrsme'] = metrics['rmse'] / np.mean(actual_value)
        metrics['dtw'] = dtw(actual_value_comp, full_pred)
        metrics['wmae'] = calculate_weighted_mae(actual_value_comp, full_pred)
        metrics['wmape'] = calculate_weighted_mape(actual_value_comp, full_pred)

        # Relative errors
        rel_metrics = {
            'relative_mape': metrics['mape'] / mean_absolute_percentage_error(actual_value_comp, restricted_pred),
            'relative_rmse': metrics['rmse'] / root_mean_squared_error(actual_value_comp, restricted_pred),
            'relative_mae': metrics['mae'] / mean_absolute_error(actual_value_comp, restricted_pred),
            'relative_wmae': metrics['wmae'] / calculate_weighted_mae(actual_value_comp, restricted_pred),
            'relative_wmape': metrics['wmape'] / calculate_weighted_mape(actual_value_comp, restricted_pred),
            'relative_dtw': metrics['dtw'] / dtw(actual_value_comp, restricted_pred),
            'relative_mse': metrics['mse'] / root_mean_squared_error(actual_value_comp, restricted_pred)**2
        }

        ssr_value = np.sum((actual_value_comp - full_pred) ** 2) if include_ssr else None
        weighted_ssr = np.sum(((actual_value_comp - full_pred)/max(actual_value_comp)) ** 2) / len(full_pred) if include_ssr else None
        
        gc_prediction_summary_tmp.loc[len(gc_prediction_summary_tmp)] = [
            dip, row[f'{cultivation_value_label}_granger_label'], opt_lag, row['max_diff'], row[f'{cultivation_value_label}_ssr_chi2test'],
            row[f'{cultivation_value_label}_significant_pvals'], metrics['mape'], metrics['mae'], metrics['rmse'],
            metrics['nrsme'], metrics['mse'], metrics['dtw'], metrics['wmae'], metrics['wmape'],
            rel_metrics['relative_mape'], rel_metrics['relative_rmse'], rel_metrics['relative_mae'],
            rel_metrics['relative_wmae'], rel_metrics['relative_wmape'], rel_metrics['relative_dtw'],
            rel_metrics['relative_mse'], dtw_mape(actual_value_comp, full_pred),
            ssr_value if include_ssr else None, 
            weighted_ssr if include_ssr else None,
            (dpis[opt_lag:], full_pred)
        ]

        # Save reference metrics for restricted prediction
        if idx == restricted_pred_index:
            for metric in ['mape', 'mae', 'rmse', 'nrsme', 'mse', 'dtw', 'wmae', 'wmape']:
                ref_ols_performance_dct_tmp[metric.upper()] = metrics[metric]
            ref_ols_performance_dct_tmp['DTW_ALIGNED_MAPE'] = dtw_mape(actual_value_comp, restricted_pred)
            if include_ssr:
                ref_ols_performance_dct_tmp['SSR'] = np.sum((actual_value_comp - restricted_pred) ** 2)
                ref_ols_performance_dct_tmp['WEIGHTED_SSR'] = np.sum(((actual_value_comp - restricted_pred)/actual_value_comp) ** 2) / len(restricted_pred)

        # Plot if dip is in plot_dips
        if dip in plot_dips:
            granger_label = row[f'{cultivation_value_label}_granger_label']
            fig, ax = plt.subplots(figsize=(figsize))
            ax.plot(dpis, actual_value, label=f'Real {cultivation_value_label}', marker='D', color='black')
            ax.plot(dpis[opt_lag:], restricted_pred, label='Restricted\nmodel prediction', marker='o', color='#e09312', linewidth=4)
            ax.plot(dpis[opt_lag:], full_pred, label=f'Full model\nprediction with\n{dip}', marker='o', color=granger_label_color_map[granger_label], linewidth=4)
            ax.set_title(f'Prediction with Granger-{row[f"{cultivation_value_label}_granger_label"]} DIP (lag={opt_lag})')
            ax.set_xlabel(xlabel)
            ax.set_ylabel(ylabel)
            ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
            ax.set_yscale(yscale)
            plt.show()
            
    # For color correction later
    gc_prediction_summary_tmp['norm_ssr_chi2test_pval'] = gc_prediction_summary_tmp['ssr_chi2test_pval'] / 0.1
    
    # Group the DataFrame by 'granger_label'
    grouped = gc_prediction_summary_tmp.groupby('granger_label')
    gc_prediction_summary_tmp['group_norm_ssr_chi2test_pval'] = gc_prediction_summary_tmp['ssr_chi2test_pval']
    # Normalize the 'norm_ssr_chi2test' column within each group
    for label, group in grouped:
        max_value = group['group_norm_ssr_chi2test_pval'].max()  # Get the maximum value in the group
        gc_prediction_summary_tmp.loc[group.index, 'group_norm_ssr_chi2test_pval'] = group['group_norm_ssr_chi2test_pval'] / max_value  # Normalize
    
    return gc_prediction_summary_tmp, ref_ols_performance_dct_tmp

## Fitting and plotting

### all

In [35]:
plt.rcParams.update({'font.size': 16})

for ds in DATASETS:
    neg[ds].update({'pred': {}, 'ref': {}})

for ds in DATASETS:
    D = neg[ds]
    for fixlag in range(1, max_fixed_lag):
        print(f'[{ds}] Running Granger prediction analysis for lag {fixlag}...')
        D['pred'][fixlag], D['ref'][fixlag] = run_granger_prediction(
            summary_df=D['summary'][fixlag],
            gc_max_lag=D['gc_max_lag'][fixlag],
            gc_test_results=D['test_results'][fixlag],
            plot_dips=[],
            log_long_data=D['stationary_ts_df'],
            restricted_pred_index=D['summary'][fixlag].iloc[0].name,
            include_ssr=True,
            figsize=(5, 3),
            ylabel='log10(PFU)',
        )


### corrected

In [36]:
for ds in DATASETS:
    neg[ds].update({'pred_corrected': {}, 'ref_corrected': {}})

for ds in DATASETS:
    D = neg[ds]
    for fixlag in range(1, max_fixed_lag):
        print(f'[{ds}] Running Granger prediction analysis with p-value correction for lag {fixlag}...')
        D['pred_corrected'][fixlag], D['ref_corrected'][fixlag] = run_granger_prediction(
            summary_df=D['summary_corrected'][fixlag],
            gc_max_lag=D['gc_max_lag'][fixlag],
            gc_test_results=D['test_results'][fixlag],
            plot_dips=[],
            log_long_data=D['stationary_ts_df'],
            restricted_pred_index=D['summary_corrected'][fixlag].iloc[0].name,
            include_ssr=True,
        )


## plot single candidates

In [37]:
def plot_single_candidate(row,
                          gc_prediction_summary_df,
                          gc_test_results,
                          log_long_data,
                          cultivation_value_label='plaque_assay',
                          dpis=None,
                          figsize=(7, 5),
                          xlabel='Time post infection (days)',
                          ylabel='log10(PFU)',
                          yscale='linear',
                          opt_lag=2,
                          title_prefix=''):
  # NOTE: the required objects are now passed in explicitly -- the old default
  # arguments were evaluated at definition time against notebook globals
  # (stationary_ts_df / summary_df[2] / ...), which no longer exist now that the
  # shuffled and random branches are kept separate.
  if dpis is None:
    dpis = log_long_data.index.values
  restricted_pred = gc_test_results[(cultivation_value_label, row.key)][row.plaque_assay_max_lag][1][0].predict()
  full_pred = gc_test_results[(cultivation_value_label, row.key)][row.plaque_assay_max_lag][1][1].predict()
  granger_label = row[f'{cultivation_value_label}_granger_label']
  actual_value = log_long_data[cultivation_value_label]
  dip = row.key
  print(gc_prediction_summary_df[gc_prediction_summary_df.key == dip].iloc[0]['ssr'])
  granger_label = row[f'{cultivation_value_label}_granger_label']

  fig, ax = plt.subplots(figsize=(figsize))
  ax.plot(dpis, actual_value, label=f'{cultivation_value_label}', marker='D', color='black')
  ax.plot(dpis[opt_lag:], restricted_pred, label='Restricted\nmodel prediction', marker='o', color='#e09312', linewidth=4)
  ax.plot(dpis[opt_lag:], full_pred, label=f'Full model\nprediction with\n{dip}', marker='o', color=granger_label_color_map[granger_label], linewidth=4)
  ax.set_title(f'{title_prefix}Prediction with Granger-{row[f"{cultivation_value_label}_granger_label"]} DIP {dip.replace("_","-")} (lag={opt_lag})')
  ax.set_xlabel(xlabel)
  ax.set_ylabel(ylabel)
  ax.legend(['$\mathregular{c_{virus}}$', 'Restricted model', 'Full model'], loc='center left', bbox_to_anchor=(1, 0.5), ncol=3)
  ax.set_yscale(yscale)
  plt.show()
  
def plot_restricted_only(row,
                          gc_prediction_summary_df,
                          gc_test_results,
                          log_long_data,
                          cultivation_value_label='plaque_assay',
                          dpis=None,
                          figsize=(7, 5),
                          xlabel='Time post infection (days)',
                          ylabel='log10(PFU)',
                          yscale='linear',
                          opt_lag=2,
                          title_prefix=''):
  if dpis is None:
    dpis = log_long_data.index.values
  restricted_pred = gc_test_results[(cultivation_value_label, row.key)][row.plaque_assay_max_lag][1][0].predict()
  actual_value = log_long_data[cultivation_value_label]
  dip = row.key

  fig, ax = plt.subplots(figsize=(figsize))
  ax.plot(dpis, actual_value, label=f'{cultivation_value_label}', marker='D', color='black')
  ax.plot(dpis[opt_lag:], restricted_pred, label='Restricted\nmodel prediction', marker='o', color='#e09312', linewidth=4)
  ax.set_title(f'{title_prefix}Restricted model prediction (lag={opt_lag})')
  ax.set_xlabel(xlabel)
  ax.set_ylabel(ylabel)
  ax.legend(['$\mathregular{c_{virus}}$', 'Restricted model'], loc='center left', bbox_to_anchor=(1, 0.5), ncol=2)
  ax.set_yscale(yscale)
  plt.show()


In [38]:
for ds in DATASETS:
    D = neg[ds]
    for fix_lag in range(1, max_fixed_lag):
        best = D['summary'][fix_lag].sort_values(by='plaque_assay_ssr_chi2test').iloc[0]
        print(f'[{ds}] lag {fix_lag}: best (lowest ssr_chi2 p-value) candidate = {best.key}')
        plot_single_candidate(
            cultivation_value_label='plaque_assay',
            dpis=D['stationary_ts_df'].index.values,
            row=best,
            figsize=(7, 5),
            xlabel='Time post infection (days)',
            ylabel='log10(PFU)',
            yscale='linear',
            opt_lag=fix_lag,
            gc_prediction_summary_df=D['pred'][fix_lag],
            gc_test_results=D['test_results'][fix_lag],
            log_long_data=D['stationary_ts_df'],
            title_prefix=f'{DATASET_TITLES[ds]}: ',
        )


In [39]:
for ds in DATASETS:
    D = neg[ds]
    for fix_lag in range(1, max_fixed_lag):
        plot_restricted_only(
            cultivation_value_label='plaque_assay',
            dpis=D['stationary_ts_df'].index.values,
            row=D['summary'][fix_lag].sort_values(by='plaque_assay_ssr_chi2test').iloc[0],
            figsize=(7, 5),
            xlabel='Time post infection (days)',
            ylabel='log10(PFU)',
            yscale='linear',
            opt_lag=fix_lag,
            gc_prediction_summary_df=D['pred'][fix_lag],
            gc_test_results=D['test_results'][fix_lag],
            log_long_data=D['stationary_ts_df'],
            title_prefix=f'{DATASET_TITLES[ds]}: ',
        )


# Swarmplots

In [40]:
import seaborn as sns

## Helper Functions

In [41]:
def get_pval_symbol(pval):
    if pval < 0.0001:
        return '****'
    elif pval < 0.001:
        return '***'
    elif pval < 0.01:
        return '**'
    elif pval < 0.05:
        return '*'
    else:
        return ''

def better_percentage(group, val):
    n = len(group)
    perc = round(len(group[group < val]) / n * 100)
    return f'{perc}%\nbetter'

def get_main_color(granger_label):
    if granger_label == 'caused':
        return 'tab:red'
    elif granger_label == 'bi-directional':
        return 'tab:purple'
    elif granger_label == 'causing':
        return 'tab:blue'
    else:
        return 'grey'

def get_corrected_color(row, column='granger_score', invert=True):
    if row['granger_label'] == 'caused':
        base_color = plt.get_cmap('Reds')
    elif row['granger_label'] == 'bi-directional':
        base_color = plt.get_cmap('Purples')
    elif row['granger_label'] == 'causing':
        base_color = plt.get_cmap('Blues')
    else:
        base_color = plt.get_cmap('Greys')

    if invert == True:
        base_color = base_color.reversed()
        
    return base_color(row[column])

# Return a lighter shade based on the granger_score (the lower the score, the lighter the shade)
def get_corrected_color_v2(row, label_column, categories=[None,None,None], column='granger_score', invert=True):
    if row[label_column] == categories[0]:
        base_color = plt.get_cmap('Reds')
    elif row[label_column] == categories[1]:
        base_color = plt.get_cmap('Purples')
    elif row[label_column] == categories[2]:
        base_color = plt.get_cmap('Blues')
    else:
        base_color = plt.get_cmap('Greys')

    if invert == False:
        base_color = base_color.reversed()
        
    # Return a lighter shade based on the granger_score (the lower the score, the lighter the shade)
    return base_color(row[column])

# Function to calculate Cohen's d
def cohens_d(group1, group2):
    # Calculate means and standard deviations
    mean1, mean2 = np.mean(group1), np.mean(group2)
    std1, std2 = np.std(group1, ddof=1), np.std(group2, ddof=1)
    
    # Pooled standard deviation
    pooled_std = np.sqrt(((len(group1) - 1) * std1 ** 2 + (len(group2) - 1) * std2 ** 2) / (len(group1) + len(group2) - 2))
    
    # Cohen's d
    return (mean1 - mean2) / pooled_std

# Function to calculate Cliff's delta
def cliffs_delta(group1, group2):
    m, n = len(group1), len(group2)
    count = sum(1 if x > y else -1 if x < y else 0 for x in group1 for y in group2)
    return count / (m * n)

def test_statistical_difference(group1, group2, alpha=0.05):
    """
    Tests if two groups are statistically different from each other and returns effect size.

    Parameters:
    - group1, group2: The two groups to be compared.
    - alpha: Significance level for the tests.

    Returns:
    - A tuple containing:
    - p-value from the test
    - Effect size (Cohen's d for t-test, Cliff's delta for Mann-Whitney U test)
    - Test type ('t-test' or 'mann-whitney')
    """
    
    # Test for normality
    shapiro_test_group1 = shapiro(group1)
    shapiro_test_group2 = shapiro(group2)
    
    if shapiro_test_group1.pvalue > alpha and shapiro_test_group2.pvalue > alpha:
        # If both groups are normal, use T-test
        t_stat, t_pvalue = ttest_ind(list(group1), list(group2))
        
        # Calculate Cohen's d
        d = cohens_d(group1, group2)
        
        return t_pvalue, d, 't-test'
    else:
        # If either group is not normal, use Mann-Whitney U test
        mw_stat, mw_pvalue = mannwhitneyu(group1, group2)
        
        # Calculate Cliff's delta
        delta = cliffs_delta(group1, group2)
        
        return mw_pvalue, delta, 'mann-whitney'



## Functions

In [42]:
effect_size_dct = {'mann-whitney': 'Cliff\'s delta', 't-test': 'Cohen\'s d'}

def make_performance_swarmplot(dip_forecast_summary,
                                ref_performance_dct,
                                performance_metric='mape',
                                forecasted_value = 'pfu',
                                title = 'Granger-related DI vRNAs predictive power over PFU\n',
                                ylabel='Mean Absolute Percentage Error (MAPE)',
                                dot_color_ref = 'diff_level_norm',
                                inverted_colors = True, # the higher the value, the darker the color
                                upper_border = -10,
                                ymax = None,
                                ymin = 0,
                                yscale = 'linear',
                                p_val_pos = 0,
                                figsize=(8, 4),
                                better_perc = False,
                                markersize=18,
                                linewidth=4,
                                dotsize=8,
                                included_dvgs = None
                                ): 
    dip_forecast_summary['corrected_color'] = dip_forecast_summary.apply(get_corrected_color, args=(dot_color_ref, inverted_colors), axis=1)
    
    if included_dvgs is not None:
        dip_forecast_summary = dip_forecast_summary[dip_forecast_summary.key.isin(included_dvgs)]
        
    # Create the swarm plot
    fig, axs = plt.subplots(nrows=1, figsize=figsize, dpi=800)
    ax = axs
    ax = sns.swarmplot(x='granger_label', 
                        y=performance_metric, 
                        hue='key',
                        data=dip_forecast_summary, 
                        order = ['causing', 'bi-directional', 'caused', 'non-related'],
                        palette=dip_forecast_summary['corrected_color'].tolist(),
                        size=dotsize, ax=ax, legend=False, linewidth=0.2
    )
    # Set title and other plot parameters
    ax.set_title(title, pad=10)
    ax.set_xlabel('Granger-causality label', labelpad=10)

    ax.set_ylabel(ylabel)
    xmin, xmax = ax.get_xlim()
    ax.hlines(y=ref_performance_dct[performance_metric.upper()], xmin=xmin, xmax=xmax, label='restricted model performance', color='#e09312', linewidth=linewidth, zorder=10)
    ax.legend(loc='lower right')
    if ymax is None:
        ymax = ref_performance_dct[performance_metric.upper()]*3
    ax.set_ylim(ymin, ymax)
    ax.set_yscale(yscale)
    #ax.invert_yaxis()
    ax.tick_params(top=False, labeltop=False, bottom=True, labelbottom=True)

    xs, xticklabels = ax.get_xticks(), ax.get_xticklabels()

    # Loop through the x-axis categories and plot the medians as markers
    for x, xticklabel in zip(xs, xticklabels):
        label = xticklabel.get_text()
        
        # Calculate median
        median_value = dip_forecast_summary[dip_forecast_summary.granger_label == label][performance_metric].median()
        
        # Plot the median as a horizontal marker (-)
        ax.plot(x, median_value, marker='+', color='black', markersize=markersize, zorder=11, label=f'{label} Median', markeredgewidth=linewidth)
        ax.plot(x, median_value, marker='+', color='yellow', markersize=markersize, zorder=1, label=f'{label} Median')
        
        ax.text(x, p_val_pos, f'n={len(dip_forecast_summary[dip_forecast_summary.granger_label == label])}', ha='center', va='bottom', color='black', zorder=10)

        if better_perc:
            ax.text(x, upper_border, 
                better_percentage(dip_forecast_summary[dip_forecast_summary.granger_label == label][performance_metric], 
                                        ref_performance_dct[performance_metric.upper()]), 
                    ha='center', va='bottom', color='black', zorder=10)
    
    fig.tight_layout()
    return fig, ax

## Per-dataset swarmplots

Each negative control gets its own swarmplot, for raw (`all`) and BH-corrected labels. Because the summaries are built per dataset, no `included_dvgs` filtering is needed — a shuffled plot can no longer silently end up empty because it was filtered against a random-derived summary.


In [43]:
label_order = ['causing', 'bi-directional', 'caused', 'non-related']
_rank = {label: i for i, label in enumerate(label_order)}

for ds in DATASETS:
    D = neg[ds]
    for fixlag in range(1, max_fixed_lag):
        for variant in ['pred', 'pred_corrected']:
            df = D[variant][fixlag]
            df = df.sort_values(by='granger_label', key=lambda x: x.map(_rank))
            # colour reference used by the swarmplot
            df['non_zero_counts'] = df['key'].map(D['nonzero_counts'])
            D[variant][fixlag] = df


In [44]:
DATASET_TITLES = {
    'shuffled': 'shuffled',
    'random': 'random'}

In [45]:
plt.rcParams.update({'font.size': 10})

# one swarmplot per dataset x lag x variant -- each summary now contains ONLY
# the DVGs of its own dataset, so no included_dvgs filtering is needed.
VARIANT_SPECS = [
    ('(A)', 'all',       'pred',           'ref',           ''),
    ('(B)', 'corrected', 'pred_corrected', 'ref_corrected', ', Benjamini-Hochberg corrected'),
]



for ds in ['shuffled']:
    D = neg[ds]
    for title_prefix, variant, pred_key, ref_key, title_suffix in VARIANT_SPECS:
        for fixlag in range(1, 2):
            print(f'[{ds}/{variant}] Creating performance swarmplot for lag {fixlag}...')
            ref = D[ref_key][fixlag]
            fig, ax = make_performance_swarmplot(
                dip_forecast_summary=D[pred_key][fixlag],
                ref_performance_dct=ref,
                performance_metric='ssr',
                forecasted_value='plaque_assay',
                title=(f'{title_prefix} Granger-causality analysis on {DATASET_TITLES[ds]} DVG time series,\n'
                      f'OLS models for log10(PFU/mL+1) (lag={fixlag}){title_suffix}'),
                ylabel='SSR',
                dot_color_ref='group_norm_ssr_chi2test_pval',
                inverted_colors=True,
                upper_border=-0.05,
                ymax=ref['SSR'] * 1.5,
                ymin=80,
                p_val_pos=ref['SSR'] * 1.2,
                figsize=(6, 3),
                dotsize=3,
                linewidth=2,
            )
            fig.savefig(f'{output_prefix}/plots/{ds}_ssr_swarmplot_{variant}_lag{fixlag}.png',
                        dpi=800, bbox_inches='tight')
            plt.show()


# Swarmplots of real data vs negative control

Each real DVG group (`causing`, `bi-directional`, `caused`, `non-related`) is
compared against a **shuffled negative control** (same series, temporal order
destroyed, run through the identical pipeline). Points are per-DVG SSR of the
full model; the `+` marks the group median and the orange line the restricted
(PFU-only) reference.

Brackets show pairwise **Mann–Whitney U** (Bonferroni-corrected) with **Cliff's
δ** for p < 0.05, matching notebook 03. Only real-vs-control comparisons are
tested; real-vs-real is omitted because the groups are defined by the same
statistic that drives SSR (circular). Shown for both uncorrected and
BH-corrected labels.

* Loads the 4 REAL Granger groups exported by notebook 01.
* Takes the SHUFFLED negative-control DVGs (computed in this notebook),
relabels them 'shuffled', and appends them as a 5th swarm.
* Runs pairwise Mann-Whitney U (Bonferroni-corrected) + Cliff's delta.
* Draws brackets ONLY for real-vs-shuffled comparisons (the statistically
valid, non-circular test) and annotates each with Cliff's delta.
Patched make_performance_swarmplot below == nb03's, plus a `granger_label_col`
/ `key_col` shim so it consumes the native nb01/nb02 schema directly.

In [46]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from scipy.stats import mannwhitneyu, kruskal
from statsmodels.stats.multitest import multipletests
from matplotlib.gridspec import GridSpec

plt.rcParams.update({'font.size': 18})

# ------------------------- CONFIG (edit here) -------------------------------
LAG                      = 1
NB01_OUTPUT_DIR          = 'data/outputs/plus1_log10_linear_imputation'  # real-data nb output_prefix
REAL_REF_CSV             = f'{NB01_OUTPUT_DIR}/ref_ols_performance_dct.csv'
REAL_LABELS              = ['causing', 'bi-directional', 'caused', 'non-related']
METRIC                   = 'ssr'
COLOR_REF                = 'group_norm_ssr_chi2test_pval'
MW_ADJUST                = 'bonferroni'   # multiple-testing correction for MWU
ALPHA                    = 0.05           # bracket shown when adjusted p < ALPHA
BRACKETS_VS_CONTROL_ONLY = True           # True => only real-vs-control brackets (avoids double-dipping)
COLOR_CONTROL_BY_LABEL   = True           # True => control dots coloured by their OWN granger label
SAVE_DIR                 = f'{output_prefix}/plots'
# The control label / DVG set / x-axis order are now derived per dataset
# ('shuffled', 'random') in the cells below, so the real data is compared
# against each negative control separately.
# ----------------------------------------------------------------------------


# ======================= stats machinery (from nb03) ========================
def _cliffs_delta(x, y):
    x = np.asarray(x); y = np.asarray(y)
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return np.nan
    ys = np.sort(y)
    less    = np.searchsorted(ys, x, side="left")
    greater = ny - np.searchsorted(ys, x, side="right")
    return float((less.sum() - greater.sum()) / (nx * ny))

def _sym_matrix(levels, vals, all_levels, fill=np.nan, diag=1.0):
    n = len(all_levels)
    M = np.full((n, n), fill, dtype=float)
    if diag is not None:
        np.fill_diagonal(M, diag)
    k = 0
    for i in range(len(levels)):
        for j in range(i + 1, len(levels)):
            li, lj = levels[i], levels[j]
            v = vals[k]; k += 1
            if li in all_levels and lj in all_levels:
                ii, jj = all_levels.index(li), all_levels.index(lj)
                M[ii, jj] = M[jj, ii] = v
    return pd.DataFrame(M, index=all_levels, columns=all_levels)

def pairwise_mwu_cliffs(df, value_col, group_col, label_order, mw_adjust='bonferroni'):
    """Pairwise Mann-Whitney U (two-sided) + Cliff's delta over the given groups.
    Returns (mannwhitney_p_adjusted, cliffs_delta) as label x label DataFrames."""
    g = df[[group_col, value_col]].dropna(subset=[value_col])
    levels = [lab for lab in label_order if lab in g[group_col].unique()]
    pairs = list(combinations(levels, 2))
    mw_p, deltas = [], []
    for a, b in pairs:
        x = g.loc[g[group_col] == a, value_col].to_numpy()
        y = g.loc[g[group_col] == b, value_col].to_numpy()
        if x.size == 0 or y.size == 0:
            mw_p.append(np.nan); deltas.append(np.nan)
        else:
            mw_p.append(mannwhitneyu(x, y, alternative="two-sided", method="auto").pvalue)
            deltas.append(_cliffs_delta(x, y))
    if mw_adjust:
        valid = [p for p in mw_p if not np.isnan(p)]
        if valid:
            _, p_adj, _, _ = multipletests(valid, method=mw_adjust)
            it = iter(p_adj)
            mw_p = [next(it) if not np.isnan(p) else np.nan for p in mw_p]
    mw_mat    = _sym_matrix(levels, mw_p,   label_order, fill=np.nan, diag=1.0)
    delta_mat = _sym_matrix(levels, deltas, label_order, fill=np.nan, diag=0.0)
    return mw_mat, delta_mat


# ============ patched make_performance_swarmplot (nb03 + schema shim) =======
_eff_symbol = {"cliffs_delta": "δ", "rank_biserial": "r_b"}

def _get_corrected_color_name(row, column, invert=True, label_key='label'):
    lab = row[label_key]                      # <-- color driven by this label column
    base = {'caused':'Reds','bi-directional':'Purples','causing':'Blues','restricted':'Oranges'}.get(lab, 'Greys')
    return base + ('_r' if invert else '')

def _get_corrected_color(row, column, invert=True, label_key='label'):
    lab = row[label_key]                      # <-- color driven by this label column
    if   lab == 'caused':          cmap = plt.get_cmap('Reds')
    elif lab == 'bi-directional':  cmap = plt.get_cmap('Purples')
    elif lab == 'causing':         cmap = plt.get_cmap('Blues')
    elif lab == 'restricted':      cmap = plt.get_cmap('Oranges')
    elif lab in ('non-related','shuffled','random','bootstrapped'): cmap = plt.get_cmap('Greys')
    else: raise ValueError(f"Unrecognized label '{lab}' for color mapping.")
    if invert:
        cmap = cmap.reversed()
    return cmap(row[column])

def _add_sig_bar_strip(ax_sig, x1, x2, y, h, text, linewidth=1, fontsize=16, bold=False):
    ax_sig.plot([x1, x1, x2, x2], [y, y+h, y+h, y], color='k',
                lw=linewidth * (2.0 if bold else 1.0), clip_on=False)
    ax_sig.text((x1+x2)/2, y+h, text, ha='center', va='bottom', fontsize=fontsize,
                fontweight=('bold' if bold else 'normal'))

def make_performance_swarmplot_with_stats(
        dip_forecast_summary, ref_performance_dct,
        performance_metric='ssr', forecasted_value='plaque_assay',
        title='', ylabel='SSR', dot_color_ref='group_norm_ssr_chi2test_pval',
        inverted_colors=True, ymax=210, ymin=80, yscale='linear', p_val_pos=0,
        figsize=(12,6), markersize=18, linewidth=3, dotsize=3,
        stats_matrix=None, stat_test_name=None, effect_size_matrix=None,
        effect_size_name=None, alpha=0.05, dpi=300,
        order=('causing','bi-directional','caused','non-related','shuffled'),
        x='label', plot_reference=True,
        granger_label_col='granger_label', key_col='key',
        color_label_col='color_label', title_offset=0.98):
    order = list(order)
    df = dip_forecast_summary.copy()
    # accept native nb01/nb02 schema
    if 'label' not in df.columns and granger_label_col in df.columns:
        df = df.rename(columns={granger_label_col: 'label'})
    if 'dvg' not in df.columns and key_col in df.columns:
        df = df.rename(columns={key_col: 'dvg'})
    df['dvg'] = df['dvg'].astype(str)

    # color label: use the dedicated column if provided, else fall back to x-axis label
    if color_label_col not in df.columns:
        df[color_label_col] = df['label']

    df = df.sort_values(by=dot_color_ref, ascending=inverted_colors)
    df['corrected_color']      = df.apply(_get_corrected_color,      args=(dot_color_ref, inverted_colors, color_label_col), axis=1)
    df['corrected_color_name'] = df.apply(_get_corrected_color_name, args=(dot_color_ref, inverted_colors, color_label_col), axis=1)

    fig = plt.figure(figsize=figsize, dpi=dpi)
    if stats_matrix is not None:
        gs = GridSpec(nrows=2, ncols=1, height_ratios=[7, 14], hspace=0.0)
        ax_sig = fig.add_subplot(gs[0]); ax = fig.add_subplot(gs[1])
    else:
        ax = fig.add_subplot(1, 1, 1)

    assert df['dvg'].is_unique, "dvg (hue) keys must be unique across all swarms"
    palette = df.set_index('dvg')['corrected_color'].to_dict()

    sns.swarmplot(x=x, y=performance_metric, hue='dvg', data=df, order=order,
                  palette=palette, size=dotsize, ax=ax, legend=False, linewidth=0.2)

    ax.set_xlabel('Granger-causality label', labelpad=10)
    ax.set_ylabel(ylabel)

    if plot_reference:
        xmin, xmax = ax.get_xlim()
        ax.hlines(y=ref_performance_dct[performance_metric.upper()], xmin=xmin, xmax=xmax,
                  label='restricted model', color='#e09312', linewidth=linewidth, zorder=10)
        ax.legend(loc='lower left', bbox_to_anchor=(0, 1.02), ncol=2, frameon=False)

    if ymax is None:
        ymax = ref_performance_dct[performance_metric.upper()] * 3
    ax.set_ylim(ymin, ymax); ax.set_yscale(yscale)
    ax.tick_params(top=False, labeltop=False, bottom=True, labelbottom=True)

    for xpos, xticklabel in zip(ax.get_xticks(), ax.get_xticklabels()):
        lab = xticklabel.get_text()
        sub = df[df['label'] == lab][performance_metric]
        if len(sub):
            ax.plot(xpos, sub.median(), marker='+', color="#FFFB00", markeredgecolor='black', markersize=markersize, markeredgewidth=3, zorder=11)
            ax.plot(xpos, sub.median(), marker='+', color='#FFFB00', markersize=markersize, zorder=15)
        ax.text(xpos, p_val_pos, f"n={len(df[df[x] == lab])}", ha='center', va='bottom', color='black', zorder=10)

    if stats_matrix is not None:
        ax_sig.set_xlim(ax.get_xlim()); ax_sig.set_ylim(0, 1); ax_sig.axis('off')
        stats_mat = stats_matrix.reindex(index=order, columns=order)
        eff_mat   = effect_size_matrix.reindex(index=order, columns=order) if effect_size_matrix is not None else None
        x_pos = {lbl: xp for lbl, xp in zip(order, ax.get_xticks())}
        pairs = []
        for i, a in enumerate(order):
            for j, b in enumerate(order):
                if j <= i:
                    continue
                p = stats_mat.loc[a, b]
                try:
                    show = np.isfinite(p) and (p < 1.0)   # print bracket for every real pair
                    bold = np.isfinite(p) and (p < alpha) # bold only when significant
                except Exception:
                    show = bold = False
                if show:
                    eff_txt = ""
                    if eff_mat is not None:
                        eff = eff_mat.loc[a, b]
                        if isinstance(eff, (int, float, np.floating)) and np.isfinite(eff):
                            sym = _eff_symbol.get(effect_size_name, effect_size_name or 'effect')
                            eff_txt = f"{sym}={eff:.2f}"
                    pairs.append((a, b, eff_txt, bool(bold)))
        if pairs:
            pairs.sort(key=lambda t: abs(x_pos[t[0]] - x_pos[t[1]]))
            base, step, h = 0.10, 0.24, 0.04
            levels_spans = []
            for a, b, eff_txt, bold in pairs:
                xa, xb = x_pos[a], x_pos[b]
                lo, hi = sorted((xa, xb))
                level = None
                for li, (u_lo, u_hi) in enumerate(levels_spans):
                    if hi < u_lo or lo > u_hi:
                        level = li; levels_spans[li] = (min(u_lo, lo), max(u_hi, hi)); break
                if level is None:
                    level = len(levels_spans); levels_spans.append((lo, hi))
                _add_sig_bar_strip(ax_sig, xa, xb, base + level*step, h, eff_txt, bold=bold)
    fig.suptitle(title, y=title_offset)
    fig.tight_layout(rect=[0, 0, 1, 1] if stats_matrix is not None else None)
    return fig, ax, df


In [47]:
# restricted-model reference line (same for every variant; from nb01)
ref_df  = pd.read_csv(REAL_REF_CSV)
ref_row = ref_df[ref_df['lag'] == LAG].iloc[0]
ref_dct = {'SSR': float(ref_row['SSR'])}

VARIANTS = [
    dict(tag='uncorrected',
         real_csv=f'{NB01_OUTPUT_DIR}/gc_prediction_summary_lag{LAG}.csv',
         ctrl_key='pred',
         title_tag='uncorrected labels'),
    dict(tag='corrected',
         real_csv=f'{NB01_OUTPUT_DIR}/gc_prediction_summary_corrected_lag{LAG}.csv',
         ctrl_key='pred_corrected',
         title_tag='BH-corrected labels'),
]

PLOT_DATA = {}   # (dataset, tag) -> dict(combined, mw, delta, order, title_tag)

for ds in DATASETS:                       # <-- real data vs EACH control separately
    D = neg[ds]
    control_label = ds
    order = REAL_LABELS + [control_label]

    for V in VARIANTS:
        # 1) real groups (nb01 export) -- colour label == x-axis label
        real = pd.read_csv(V['real_csv'])
        real = real[real['granger_label'].isin(REAL_LABELS)].copy()
        real['color_label'] = real['granger_label']

        # 2) negative control (this notebook, in memory) -> relabel to the control name
        ctrl = D[V['ctrl_key']][LAG].copy()
        ctrl['color_label'] = ctrl['granger_label'] if COLOR_CONTROL_BY_LABEL else control_label
        ctrl['granger_label'] = control_label              # x-axis position
        ctrl['key'] = f'ctrl_{ds}__' + ctrl['key'].astype(str)   # keep hue keys unique
        _m = ctrl[COLOR_REF].max()                        # self-consistent shading
        if _m and np.isfinite(_m) and _m > 0:
            ctrl[COLOR_REF] = ctrl[COLOR_REF] / _m

        # 3) combine on the swarmplot schema
        keep = ['key', 'granger_label', 'color_label', METRIC, COLOR_REF, 'ssr_chi2test_pval']
        combined = pd.concat([real[keep], ctrl[keep]], ignore_index=True).dropna(subset=[METRIC])
        combined = combined[combined['granger_label'].isin(order)].copy()

        # 4) pairwise MWU (Bonferroni) + Cliff's delta  <-- the slow part
        mw, delta = pairwise_mwu_cliffs(
            combined.rename(columns={'granger_label': 'label'}),
            value_col=METRIC, group_col='label', label_order=order, mw_adjust=MW_ADJUST)
        if BRACKETS_VS_CONTROL_ONLY:
            rmask = mw.index != control_label
            cmask = mw.columns != control_label
            mw.loc[rmask, cmask] = 1.0     # suppress real-vs-real brackets (circular)

        PLOT_DATA[(ds, V['tag'])] = dict(combined=combined, mw=mw, delta=delta,
                                          order=order, title_tag=V['title_tag'])
        print(f'prepared: {ds} / {V["tag"]}  (n={len(combined)})')


In [48]:
PLOT_DATA[('shuffled', 'uncorrected')]['combined'].groupby('granger_label')['ssr_chi2test_pval'].agg(min_pval='min', max_pval='max').reset_index()

In [ ]:
PLOT_VARIANTS = ['uncorrected', 'corrected']   # e.g. ['corrected'] to redraw just one
PLOT_DATASETS = DATASETS                       # e.g. ['shuffled'] to redraw just one
title_dct = {'uncorrected': 
              f"(A) Sum of squared residuals (SSR) of OLS model fits (lag=1), grouped by Granger-causality", 
            'corrected': 
              "(B) Sum of squared residuals (SSR) of OLS model fits (lag=1), Benjamini-Hochberg-corrected"}

figs = {}


plt.rcParams.update({'font.size': 16})
#for ds in PLOT_DATASETS:
for ds in ['shuffled']:
    for tag in PLOT_VARIANTS:
        P = PLOT_DATA[(ds, tag)]
        fig, ax, PLOT_DATA[(ds, tag)]['plot_df'] = make_performance_swarmplot_with_stats(
            P['combined'], ref_performance_dct=ref_dct,
            performance_metric=METRIC, forecasted_value='plaque_assay',
            title=title_dct[tag],
            ylabel='SSR', dot_color_ref=COLOR_REF, inverted_colors=True,
            ymin=80,
            ymax=ref_dct['SSR'] * 1.3, p_val_pos=ref_dct['SSR'] * 1.1,
            figsize=(12, 8), dotsize=5, linewidth=3, dpi=800,
            stats_matrix=P['mw'], stat_test_name='mannwhitney_p',
            effect_size_matrix=P['delta'], effect_size_name='cliffs_delta',
            alpha=ALPHA, order=P['order'], plot_reference=True,
            color_label_col='color_label', title_offset=0.95)

        out_png = f"{SAVE_DIR}/real_vs_{ds}_{tag}_ssr_swarmplot_lag{LAG}.png"
        fig.savefig(out_png, dpi=800, bbox_inches='tight')
        figs[(ds, tag)] = fig
        print("saved:", out_png)


In [ ]:
PLOT_VARIANTS = ['uncorrected', 'corrected']   # e.g. ['corrected'] to redraw just one
PLOT_DATASETS = DATASETS                       # e.g. ['shuffled'] to redraw just one
title_dct = {'uncorrected': 
              f"Sum of squared residuals (SSR) of OLS model fits (lag=1), grouped by Granger-causality", 
            'corrected': 
              "Sum of squared residuals (SSR) of OLS model fits (lag=1), Benjamini-Hochberg-corrected"}

figs = {}

plt.rcParams.update({'font.size': 18})
#for ds in PLOT_DATASETS:
for ds in ['shuffled']:
    for tag in ['corrected']:
        P = PLOT_DATA[(ds, tag)]
        fig, ax, PLOT_DATA[(ds, tag)]['plot_df'] = make_performance_swarmplot_with_stats(
            P['combined'], ref_performance_dct=ref_dct,
            performance_metric=METRIC, forecasted_value='plaque_assay',
            title=title_dct[tag],
            ylabel='SSR', dot_color_ref=COLOR_REF, inverted_colors=True,
            ymin=80,
            ymax=200, p_val_pos=ref_dct['SSR'] * 1.1,
            figsize=(15, 6.5), dotsize=5, linewidth=3, dpi=800,
            stats_matrix=P['mw'], stat_test_name='mannwhitney_p',
            effect_size_matrix=P['delta'], effect_size_name='cliffs_delta',
            alpha=ALPHA, order=P['order'], plot_reference=True,
            color_label_col='color_label', title_offset=0.95)

        out_png = f"{SAVE_DIR}/real_vs_{ds}_{tag}_ssr_swarmplot_lag{LAG}.png"
        fig.savefig(out_png, dpi=800, bbox_inches='tight')
        figs[(ds, tag)] = fig
        print("saved:", out_png)


In [ ]:
dvg2color_df = pd.concat([PLOT_DATA[('random', 'corrected')]['plot_df'].copy(),
                          PLOT_DATA[('shuffled', 'corrected')]['plot_df'].copy()], ignore_index=True)
dvg2color_df['dvg'] = dvg2color_df['dvg'].astype(str)
dvg2color_df['dvg'] = dvg2color_df['dvg'].apply(lambda x: x.replace('ctrl_shuffled__', ''))
dvg2color_df['dvg'] = dvg2color_df['dvg'].apply(lambda x: x.replace('ctrl_random__', ''))
dvg2color_df = dvg2color_df[['dvg','label','corrected_color_name', 'corrected_color']]
dvg2color_df.to_csv(f"{SAVE_DIR}/dvg2color_shuffled_corrected.csv", index=True)
print(f"saved: {SAVE_DIR}/dvg2color_shuffled_corrected.csv")
dvg2color_df

In [ ]:
dvg2color_df[dvg2color_df['label'].isin(['causing','bi-directional','caused','non-related','shuffled'])].sort_values(by='label').to_csv(f"{output_prefix}/dvg_labels_shuffled.csv", index=False)

### Control SSR summary


In [ ]:
# median control SSR per dataset / variant
rows = []
for ds in DATASETS:
    for variant, key in [('uncorrected', 'pred'), ('corrected', 'pred_corrected')]:
        s = neg[ds][key][LAG]['ssr']
        rows.append({'dataset': ds, 'variant': variant, 'n': int(s.notna().sum()),
                      'median_ssr': s.median(), 'min_ssr': s.min(), 'max_ssr': s.max()})
control_ssr_summary = pd.DataFrame(rows)
print(control_ssr_summary.to_string(index=False))


In [ ]:
CAUSAL    = ['causing', 'bi-directional']
NONCAUSAL = ['caused', 'non-related']     # swap to ['non-related'] etc. as you like

def ssr_bounds(df, dataset, variant):
    causal    = df[df['granger_label'].isin(CAUSAL)]
    noncausal = df[df['granger_label'].isin(NONCAUSAL)]
    out = {'dataset': dataset, 'variant': variant,
           'worst_causal_ssr': np.nan, 'worst_causal_dvg': None,
           'best_noncausal_ssr': np.nan, 'best_noncausal_dvg': None}
    if len(causal) and causal['ssr'].notna().any():
        out['worst_causal_ssr'] = causal['ssr'].max()
        out['worst_causal_dvg'] = causal.loc[causal['ssr'].idxmax(), 'key']
    if len(noncausal) and noncausal['ssr'].notna().any():
        out['best_noncausal_ssr'] = noncausal['ssr'].min()
        out['best_noncausal_dvg'] = noncausal.loc[noncausal['ssr'].idxmin(), 'key']
    return out

bounds = pd.DataFrame([
    ssr_bounds(neg[ds][key][LAG], ds, variant)
    for ds in DATASETS
    for variant, key in [('uncorrected', 'pred'), ('corrected', 'pred_corrected')]
])
print(bounds.to_string(index=False))


# Group sizes with increasing lag


In [ ]:
plt.rcParams.update({'font.size': 16})

LABEL_COLORS = {'causing': 'tab:blue', 'bi-directional': 'tab:purple',
                'caused': 'tab:red', 'non-related': 'gray'}

for ds in DATASETS:
  fig, ax = plt.subplots(figsize=(8, 5))
  D = neg[ds]
  for label, color in LABEL_COLORS.items():
      sizes      = [(D['pred'][l]['granger_label'] == label).sum() for l in range(1, max_fixed_lag)]
      sizes_corr = [(D['pred_corrected'][l]['granger_label'] == label).sum() for l in range(1, max_fixed_lag)]
      ax.plot(range(1, max_fixed_lag), sizes, marker='o', color=color,
              linestyle='solid', label=label)
      ax.plot(range(1, max_fixed_lag), sizes_corr, marker='D', color=color,
              linestyle='dashed', label=f'{label} (corrected)')
  ax.set_xlabel('Fixed lag used in Granger-causality test')
  ax.set_ylabel('Number of DVGs')
  ax.set_yscale('log')
  ax.set_xticks(range(1, max_fixed_lag))
  ax.set_title(DATASET_TITLES[ds])
  ax.legend(loc='upper left', bbox_to_anchor=(1, 1), ncol=1)
  ax.set_title(f'Number of Granger-related DVGs\nacross fixed lags')
  #ax.set_title(f'[{DATASET_TITLES[ds]}] Number of Granger-related DVGs across fixed lags (solid = raw, dashed = BH-corrected)')
  plt.tight_layout()
  fig.savefig(f'{output_prefix}/plots/{ds}_granger_dvg_counts_across_lags.png', dpi=200, bbox_inches='tight')
  plt.show()


# SSR with increasing lag


In [ ]:
plt.rcParams.update({'font.size': 16})

LABEL_COLORS = {'causing': 'tab:blue', 'bi-directional': 'tab:purple',
                'caused': 'tab:red', 'non-related': 'gray'}

for variant, pred_key, ref_key, vlabel in [
        ('all',       'pred',           'ref',           'raw labels'),
        ('corrected', 'pred_corrected', 'ref_corrected', 'BH-corrected labels')]:

    for ds in DATASETS:
        fig, ax = plt.subplots(figsize=(8, 5))
        D = neg[ds]

        for fixlag in range(1, max_fixed_lag):
            ax.scatter(fixlag, D[ref_key][fixlag]['SSR'], color='tab:orange',
                       label='restricted model' if fixlag == 1 else '')

        for label, color in LABEL_COLORS.items():
            for fixlag in range(1, max_fixed_lag):
                subset = D[pred_key][fixlag]
                subset = subset[subset['granger_label'] == label]
                if len(subset):
                    ax.scatter(fixlag, subset['ssr'].median(), color=color,
                               label=label if fixlag == 1 else '', linewidth=2)

        ax.set_xlabel('Fixed lag used in Granger-causality test')
        ax.set_ylabel('Median SSR')
        ax.set_xticks(range(1, max_fixed_lag))
        ax.set_title(f'{DATASET_TITLES[ds]}\nMedian SSR vs. fixed lag ({vlabel})')
        ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))

        plt.tight_layout()
        fig.savefig(f'{output_prefix}/plots/{ds}_median_ssr_vs_lag_{variant}.png',
                    dpi=200, bbox_inches='tight')
        plt.show()

In [ ]:
plt.rcParams.update({'font.size': 16})

for variant, pred_key, ref_key, vlabel in [
        ('all',       'pred',           'ref',           'raw labels'),
        ('corrected', 'pred_corrected', 'ref_corrected', 'BH-corrected labels')]:

    for ds in DATASETS:
        fig, ax = plt.subplots(figsize=(8, 5))
        D = neg[ds]

        for fixlag in range(1, max_fixed_lag):
            ax.scatter(fixlag, D[pred_key][fixlag]['ssr'].median(),
                       color='tab:green',
                       label='full models (median)' if fixlag == 1 else '', linewidth=2)
            ax.scatter(fixlag, D[ref_key][fixlag]['SSR'],
                       color='tab:orange',
                       label='restricted model' if fixlag == 1 else '')

        ax.set_xlabel('Fixed lag used in Granger-causality test')
        ax.set_ylabel('Median SSR')
        ax.set_xticks(range(1, max_fixed_lag))
        ax.set_title(f'{DATASET_TITLES[ds]}\nMedian SSR vs. fixed lag ({vlabel})')
        ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))

        plt.tight_layout()
        fig.savefig(f'{output_prefix}/plots/{ds}_full_restricted_ssr_vs_lag_{variant}.png',
                    dpi=200, bbox_inches='tight')
        plt.show()